# Step 1: Data Loading and normalization

In [9]:
# Install sklearn if not already installed
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [10]:
# Install pywt if not already installed
%pip install PyWavelets

Note: you may need to restart the kernel to use updated packages.


In [11]:
# Install tensorflow if not already installed
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np

# Paths to your data folders
# Make sure you are running this notebook in the same directory as the data folders
truth_path = 'Truth_Sessions/1_BandPass_Filtered/'
lie_path = 'Lie_Sessions/1_BandPass_Filtered/'

# Function to load all CSVs and concatenate them into a single DataFrame
def load_and_concatenate_data(path):
    data_frames = []
    for filename in os.listdir(path):
        if filename.endswith('.csv'):
            df = pd.read_csv(os.path.join(path, filename))
            data_frames.append(df)
    concatenated_df = pd.concat(data_frames, ignore_index=True)
    return concatenated_df
#print("Current directory:", os.getcwd())
truth_data = load_and_concatenate_data(truth_path)
lie_data = load_and_concatenate_data(lie_path)

# Function to normalize data
scaler = StandardScaler()
def normalize_data(df):
    return scaler.fit_transform(df)

truth_data = normalize_data(truth_data)
lie_data = normalize_data(lie_data)



# Step 2: Segmenting the Data

In [13]:
# Function to segment data
def segment_data(df, window_size=128, overlap=64):
    segments = []
    for start in range(0, len(df) - window_size, overlap):
        segment = df[start:start + window_size, :]
        segments.append(segment)
    return np.array(segments)

# Segment the data
window_size = 128  # 1 second of data
overlap = 64  # 50% overlap

truth_segments = segment_data(truth_data, window_size, overlap)
lie_segments = segment_data(lie_data, window_size, overlap)

# Step 3: Feature Extraction with DWT

In [14]:
import pywt

# Function to extract DWT features
def extract_dwt_features(segments, wavelet='db4', level=4):
    features = []
    for segment in segments:
        segment_features = []
        for channel in range(segment.shape[1]):
            coeffs = pywt.wavedec(segment[:, channel], wavelet, level=level)
            coeffs_flat = np.hstack(coeffs)
            segment_features.append(coeffs_flat)
        features.append(np.hstack(segment_features))
    return np.array(features)

# Extract DWT features
truth_features = extract_dwt_features(truth_segments)
lie_features = extract_dwt_features(lie_segments)

# Step 4: Preparing Data for CNN

In [15]:
# Create labels: 1 for truth, 0 for lie
truth_labels = np.ones(truth_features.shape[0])
lie_labels = np.zeros(lie_features.shape[0])

# Combine features and labels
X = np.vstack((truth_features, lie_features))
y = np.hstack((truth_labels, lie_labels))

# Split data into training and testing sets
from sklearn.model_selection import train_test_split

# Ensure both classes are in the test set by using stratified split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Verify the split
print("Training set class distribution:", np.bincount(y_train.astype(int)))
print("Test set class distribution:", np.bincount(y_test.astype(int)))

Training set class distribution: [3238 3238]
Test set class distribution: [810 810]


# Step 5: Define and Train the CNN

In [16]:
import tensorflow as tf
from tensorflow.keras import layers, models,callbacks

# Define the CNN model
model = models.Sequential()

# Convolution Layers
# Stage 1
model.add(layers.Conv1D(256, 3, activation='relu', input_shape=(X_train.shape[1], 1)))
model.add(tf.keras.layers.BatchNormalization())
model.add(layers.MaxPooling1D(2))
model.add(layers.Dropout(0.25))

# Stage 2
model.add(layers.Conv1D(128, 3, activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(layers.MaxPooling1D(2))
model.add(layers.Dropout(0.25))

# Stage 3
model.add(layers.Conv1D(64, 3, activation='relu'))
model.add(tf.keras.layers.BatchNormalization())
model.add(layers.MaxPooling1D(2))
model.add(layers.Dropout(0.25))

# Flatten
model.add(layers.Flatten())

# Fully Connected Layers
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(2, activation='softmax'))

# Output
model.add(layers.Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Reshape data for the CNN
X_train_cnn = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

# Define callbacks for early stopping and model checkpointing
callbacks = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    callbacks.ModelCheckpoint('best_model.keras', save_best_only=True)
]

# Train the model
history = model.fit(X_train_cnn, y_train, epochs=10, batch_size=32, validation_data=(X_test_cnn, y_test), callbacks=callbacks)

# Load the best model
model.load_weights('best_model.keras')

2026-02-11 14:22:23.384119: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


RuntimeError: This version of jaxlib was built using AVX instructions, which your CPU and/or operating system do not support. This error is frequently encountered on macOS when running an x86 Python installation on ARM hardware. In this case, try installing an ARM build of Python. Otherwise, you may be able work around this issue by building jaxlib from source.

# Step 6: Evaluate the Model

In [ ]:

from sklearn.metrics import classification_report, confusion_matrix, f1_score
# Load the best model
model = models.Sequential()
model.load_weights('best_model.keras')
# Make predictions
y_pred = model.predict(X_test_cnn)
y_pred_classes = (y_pred > 0.5).astype("int32")

# Evaluate the model
print(classification_report(y_test, y_pred_classes, zero_division=1))
print("F1 Score:", f1_score(y_test, y_pred_classes, zero_division=1))


NameError: name 'models' is not defined